In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
Using cached sympy-1.14.0-py3-none-any.whl

In [26]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

#Creating the entangled qbit to distribute to Bob and Alice
def entangled_bits():
    circuit = QuantumCircuit(2,2)
    circuit.h(0)
    circuit.cx(0,1)
    circuit.z(1)
    circuit.x(0)
    return circuit


# Creating a qbit to generate a random number between 1 and 3
def quantum_random_generator():
    random_qbit = QuantumCircuit(2,2)

    t_matrix = [ [ (1/math.sqrt(3)) , (math.sqrt(2)/math.sqrt(3))],
          [ ((-1) * (math.sqrt(2)/math.sqrt(3))) , (1/math.sqrt(3))] ]
    
    t_operator = Operator(t_matrix)
    
    random_qbit.unitary(t_operator, [0])
    random_qbit.h(1)
    
    random_qbit.measure(range(2), range(2))
        
    backend = BasicSimulator()
    qc_compiled = transpile(random_qbit, backend)
    job_sim = backend.run(qc_compiled, shots = 1)
    result_sim = job_sim.result()
    result = result_sim.get_counts(qc_compiled)
    
    result = list(result)[0]
    
    if result[1] == '0':
        return 1
    elif result[0] == '1':
        return 2
    else:
        return 3

# Function to execute the Ekert protocol
def execute_protocol(n):

    # Operator matrices for the w and v states 
    root2 = math.sqrt(2)
    denom1 = math.sqrt(4 + 2*root2)
    denom2 = math.sqrt(4 - 2*root2) 
    
    w_operator_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                            [  1 / denom2 , (root2 - 1) / denom2 ] ]
    w_operator = Operator(w_operator_matrix) 
    
    
    v_operator_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                            [ -1 / denom2 , (root2 - 1) / denom2 ] ]
    v_operator = Operator(v_operator_matrix)
    

    # Converter and inverter
    converter = {'0': 1, '1': -1}
    inverter = {'0': '1', '1': '0'}
    
    final_key = ""

    # total value of each pairing of unitary operation and number of times each pairing has occurred, for averaging
    s_value = {"xw": 0, "xv": 0, "zw": 0, "zv": 0}
    s_times = {"xw": 0, "xv": 0, "zw": 0, "zv": 0}
    
    for i in range(0, int((9*n)/2)):

        # Create the entangled bit
        circuit = entangled_bits()
        
        # Alice chooses her operator
        alice_operator_num = quantum_random_generator()
        
        # Bob chooses his operator
        bob_operator_num = quantum_random_generator()

        # Alice applies the appropriate unitary operator
        if alice_operator_num == 1:
            circuit.h(0)
        elif alice_operator_num == 2:
            circuit.unitary(w_operator, [0])
        else:
            pass    

        # Bob applies the appropriate unitary operator
        if bob_operator_num == 1:
            circuit.unitary(w_operator, [1])
        elif bob_operator_num == 2:
            pass
        else:
            circuit.unitary(v_operator, [1])
            
        # Both bits are measured
        circuit.measure(range(2), range(2))
        
        backend = BasicSimulator()
        qc_compiled = transpile(circuit, backend)
        job_sim = backend.run(qc_compiled, shots = 1)
        result_sim = job_sim.result()
        result = result_sim.get_counts(qc_compiled)
    
        result = list(result)[0]
        alice_bit = converter[result[1]]
        bob_bit = converter[inverter[result[0]]]
    
        #Bob and Alice compare operator choices and either add to the shared key, or add to the counts for averaging for s calculation
        if alice_operator_num == 1 and bob_operator_num == 1:
            s_times["xw"] += 1
            s_value["xw"] += (alice_bit * bob_bit)
        elif alice_operator_num == 1 and bob_operator_num == 3:
            s_times["xv"] += 1
            s_value["xv"] += (alice_bit * bob_bit)
        elif alice_operator_num == 3 and bob_operator_num == 1:
            s_times["zw"] += 1
            s_value["zw"] += (alice_bit * bob_bit)
        elif alice_operator_num == 3 and bob_operator_num == 3:
            s_times["zv"] += 1
            s_value["zv"] += (alice_bit * bob_bit)
        elif alice_operator_num == 2 and bob_operator_num == 1:
            final_key += result[0] 
        elif alice_operator_num == 3 and bob_operator_num == 2:
            final_key += result[0] 
        else:
            pass

    # Calculates s, with contingency for a division by 0 error
    for k in s_value.keys():
        if s_times[k] == 0:
            s_value[k] = 0
        else:    
            s_value[k] = (s_value[k]/s_times[k])
    
    s = abs(s_value["xw"] - s_value["xv"] + s_value["zw"] + s_value["zv"])
    
    print("Final key is: " + final_key)
    print("S value is: " + str(s))

#Execute protocol for a key of around length n
execute_protocol(500)


Final key is: 1010000110000101010110010100010011111100001101001101001011001101010101101111011110101100011011110001011000100101110010001111100111110111100000110000010110100100101011110100110010001111010001110100100101010110101011111010101100101001010110100111101000111001100011000001110010000011111010001001001101100010110111110011000011101101101110111011001101100000111100001010001010111000011011110001101011110010101110100101001101110001111111011001011000010010101001011011111011000011110110011001101001001111111010110110111100000011001010111001011000011110001000100101101001010101101000001111000011000001000111010001110000101100100010100001000101100101110011110001101101001000011001001010000010101010001111001001101001110011101011111000010110111110011110010011001011010101010110111010001010110101101010100101100000011000100010101011001111110000011000111101100010101010011101101110100100001010010000010101111101000010010010100010110110111111110011001011011011011011100101010100010001001111
S value i